# Automatic Denoising (Gavish-Donoho Method)

SVD is a powerful tool for separating signal from noise. The challenge lies in choosing the correct rank  to filter out the noise while preserving the data.

This example uses the **Gavish-Donoho optimal threshold** to automatically determine the optimal cut-off rank , allowing `rsvd` to reconstruct the clean signal without manual tuning.

In [8]:
import numpy as np
from randomized_svd.utils import optimal_threshold
from randomized_svd.core import rsvd

# Set seed for reproducibility
np.random.seed(42)

# 1. Create Clean Signal (Low Rank)
n = 1000
true_rank = 20
U_true, _ = np.linalg.qr(np.random.randn(n, true_rank))
V_true, _ = np.linalg.qr(np.random.randn(n, true_rank))
S_true = np.diag(np.linspace(10, 5, true_rank))
X_clean = U_true @ S_true @ V_true.T

# 2. Add Gaussian Noise
noise_level = 0.5
X_noisy = X_clean + noise_level * np.random.randn(n, n)

# 3. Detect Optimal Rank automatically
# The Gavish-Donoho method finds the optimal cutoff based on matrix size and noise level
opt_k = optimal_threshold(n, n, gamma=noise_level)
print(f"True Rank: {true_rank}")
print(f"Detected Optimal Rank: {opt_k}")

# 4. Filter (Denoise)
U, S, Vt = rsvd(X_noisy, t=opt_k)
X_denoised = U @ S @ Vt

# 5. Measure improvement
err_noisy = np.linalg.norm(X_clean - X_noisy)
err_denoised = np.linalg.norm(X_clean - X_denoised)

print(f"\nOriginal Error:  {err_noisy:.2f}")
print(f"Denoised Error:  {err_denoised:.2f}")
print(f"Noise Reduction: {((err_noisy - err_denoised) / err_noisy) * 100:.1f}%")

True Rank: 20
Detected Optimal Rank: 37

Original Error:  500.14
Denoised Error:  142.15
Noise Reduction: 71.6%
